# DeepShield — Treinamento no Google Colab
**Detector de Deepfakes** baseado em EfficientNet-B0 com transfer learning.

**Estratégia:**
- **Fase 1** (5 epochs): backbone congelado, treina apenas o classificador
- **Fase 2** (10 epochs): fine-tune dos últimos 3 blocos com LR 10× menor

> ⚠️ **Ative a GPU:** Runtime → Change runtime type → GPU

## 1. Verificar GPU

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU disponivel: {gpu_name} ({gpu_mem:.1f} GB)")
    DEVICE = torch.device("cuda")
else:
    print("GPU nao encontrada. Ative em: Runtime > Change runtime type > GPU")
    DEVICE = torch.device("cpu")

print(f"PyTorch: {torch.__version__}")
print(f"Device : {DEVICE}")

## 2. Clonar o repositório

In [ ]:
!rm -rf deepshield
!git clone https://github.com/GuizanzotiSB/deepshield.git
%cd deepshield
!ls

## 3. Instalar dependências

In [ ]:
!pip install -q -r requirements.txt

## 4. Upload do dataset

Faça upload do arquivo `.zip` do dataset (140k Real and Fake Faces do Kaggle).
A estrutura esperada dentro do zip:
```
real_vs_fake/real-vs-fake/train/real/
real_vs_fake/real-vs-fake/train/fake/
```

In [ ]:
from google.colab import files
import zipfile, os

print("Selecione o .zip do dataset:")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f"Extraindo {zip_name}...")

os.makedirs("data/raw", exist_ok=True)
with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall("data/raw/")

os.remove(zip_name)

# Verificar estrutura
for split in ("train", "valid", "test"):
    for cls in ("real", "fake"):
        p = f"data/raw/real_vs_fake/real-vs-fake/{split}/{cls}"
        n = len(os.listdir(p)) if os.path.isdir(p) else 0
        print(f"  {p}: {n} imagens")

## 5. Preparar dataset e modelo

In [ ]:
import sys, json
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Subset

from src.dataset import DeepfakeDataset, build_datasets, get_train_transforms, get_eval_transforms
from src.model import DeepShieldModel
from src.train import (
    train_one_epoch, evaluate, compute_class_weights,
    TrainHistory, auto_device,
)

DEVICE = auto_device("auto")
print(f"Device: {DEVICE}")

# Datasets (80/20 split)
train_subset, val_subset = build_datasets(val_ratio=0.2, image_size=224, seed=42)
print(f"Train: {len(train_subset)} | Val: {len(val_subset)}")

# Class weights
class_weights = compute_class_weights(train_subset).to(DEVICE)
print(f"Class weights: {class_weights.tolist()}")
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Modelo
model = DeepShieldModel(pretrained=True).to(DEVICE)
model.summary()

# Historico
history = TrainHistory()
best_f1 = -1.0
SAVE_DIR = Path("models")
SAVE_DIR.mkdir(exist_ok=True)

## 6. Fase 1 — Backbone congelado (5 epochs)
Apenas o classificador `Linear(1280,512) → ReLU → Dropout → Linear(512,2)` é treinado.

In [ ]:
PHASE1_EPOCHS = 5
PHASE1_BS = 64
PHASE1_LR = 1e-3

model.freeze_backbone()
print(model.count_parameters())

train_loader = DataLoader(train_subset, batch_size=PHASE1_BS, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=PHASE1_BS, shuffle=False, num_workers=2, pin_memory=True)

optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=PHASE1_LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS)

for epoch in range(1, PHASE1_EPOCHS + 1):
    print(f"\n[Fase 1] Epoch {epoch}/{PHASE1_EPOCHS}")

    train_m = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, desc=f"F1 E{epoch} train")
    val_m = evaluate(model, val_loader, criterion, DEVICE, desc=f"F1 E{epoch} val")
    scheduler.step()

    lr = optimizer.param_groups[0]["lr"]
    history.log(train_m, val_m, lr, phase=1)

    print(f"  train -> loss={train_m.loss:.4f} acc={train_m.accuracy:.4f} f1={train_m.f1:.4f}")
    print(f"  val   -> loss={val_m.loss:.4f} acc={val_m.accuracy:.4f} prec={val_m.precision:.4f} rec={val_m.recall:.4f} f1={val_m.f1:.4f}")

    if val_m.f1 > best_f1:
        best_f1 = val_m.f1
        torch.save(model.state_dict(), SAVE_DIR / "best_model.pth")
        print(f"  -> novo melhor F1={best_f1:.4f} salvo!")

print(f"\nFase 1 concluida. Melhor F1: {best_f1:.4f}")

## 7. Fase 2 — Fine-tune dos últimos 3 blocos (10 epochs)
Descongela os últimos 3 blocos do EfficientNet + `conv_head`/`bn2`. LR 10x menor, batch menor.

In [ ]:
PHASE2_EPOCHS = 10
PHASE2_BS = 32
PHASE2_LR = 1e-4
PATIENCE = 5

model.unfreeze_last_blocks(n=3)
print(model.count_parameters())

train_loader = DataLoader(train_subset, batch_size=PHASE2_BS, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_subset, batch_size=PHASE2_BS, shuffle=False, num_workers=2, pin_memory=True)

optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=PHASE2_LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS)

epochs_no_improve = 0

for epoch in range(1, PHASE2_EPOCHS + 1):
    print(f"\n[Fase 2] Epoch {epoch}/{PHASE2_EPOCHS}")

    train_m = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE, desc=f"F2 E{epoch} train")
    val_m = evaluate(model, val_loader, criterion, DEVICE, desc=f"F2 E{epoch} val")
    scheduler.step()

    lr = optimizer.param_groups[0]["lr"]
    history.log(train_m, val_m, lr, phase=2)

    print(f"  train -> loss={train_m.loss:.4f} acc={train_m.accuracy:.4f} f1={train_m.f1:.4f}")
    print(f"  val   -> loss={val_m.loss:.4f} acc={val_m.accuracy:.4f} prec={val_m.precision:.4f} rec={val_m.recall:.4f} f1={val_m.f1:.4f}")

    if val_m.f1 > best_f1:
        best_f1 = val_m.f1
        torch.save(model.state_dict(), SAVE_DIR / "best_model.pth")
        print(f"  -> novo melhor F1={best_f1:.4f} salvo!")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        print(f"  sem melhora ({epochs_no_improve}/{PATIENCE})")
        if epochs_no_improve >= PATIENCE:
            print("  Early stopping!")
            break

# Salvar historico
history.save(SAVE_DIR / "history.json")
print(f"\nTreinamento concluido. Melhor F1: {best_f1:.4f}")

## 8. Métricas e gráficos

In [ ]:
import matplotlib.pyplot as plt

with open(SAVE_DIR / "history.json") as f:
    hist = json.load(f)

epochs_range = range(1, len(hist["train"]) + 1)
train_loss = [e["loss"] for e in hist["train"]]
val_loss   = [e["loss"] for e in hist["val"]]
train_acc  = [e["accuracy"] for e in hist["train"]]
val_acc    = [e["accuracy"] for e in hist["val"]]
train_f1   = [e["f1"] for e in hist["train"]]
val_f1     = [e["f1"] for e in hist["val"]]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Fase 1/2 background
phase1_end = sum(1 for p in hist["phase"] if p == 1)
for ax in axes:
    ax.axvspan(0.5, phase1_end + 0.5, alpha=0.08, color="blue", label="Fase 1")
    ax.axvspan(phase1_end + 0.5, len(hist["train"]) + 0.5, alpha=0.08, color="orange", label="Fase 2")

# Loss
axes[0].plot(epochs_range, train_loss, "o-", label="Train")
axes[0].plot(epochs_range, val_loss, "o-", label="Val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, train_acc, "o-", label="Train")
axes[1].plot(epochs_range, val_acc, "o-", label="Val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# F1
axes[2].plot(epochs_range, train_f1, "o-", label="Train")
axes[2].plot(epochs_range, val_f1, "o-", label="Val")
axes[2].set_title("F1 Score")
axes[2].set_xlabel("Epoch")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

fig.suptitle(f"DeepShield — Melhor Val F1: {best_f1:.4f}", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Tabela resumo
print(f"\n{'Epoch':>5} {'Phase':>5} {'T Loss':>8} {'V Loss':>8} {'V Acc':>7} {'V Prec':>7} {'V Rec':>7} {'V F1':>7} {'LR':>10}")
print("-" * 75)
for i in range(len(hist["train"])):
    t, v = hist["train"][i], hist["val"][i]
    print(f"{i+1:>5} {hist['phase'][i]:>5} {t['loss']:>8.4f} {v['loss']:>8.4f} {v['accuracy']:>7.4f} {v['precision']:>7.4f} {v['recall']:>7.4f} {v['f1']:>7.4f} {hist['lr'][i]:>10.6f}")

## 9. Download do modelo treinado

In [ ]:
from google.colab import files

model_path = "models/best_model.pth"
history_path = "models/history.json"

print(f"best_model.pth: {Path(model_path).stat().st_size / 1e6:.1f} MB")
files.download(model_path)
files.download(history_path)
print("Downloads iniciados!")